# Week 3: Classes, complexity, searching and sorting

**Budget:** about 10 hours.

| Session | Topic | Time |
|---|---|---|
| Day 1 | Classes and dataclasses: build a `Vocabulary` | 2.5 h |
| Day 2 | Big-O by experiment, and choosing the right data structure | 2.5 h |
| Day 3 | Binary search, insertion sort, and randomised testing | 2.5 h |
| Day 4 | Package layout: turn `Vocabulary` into a tested module | 2 h |
| Review | Spaced recall, blank-page rebuild, commit | 0.5 h |

### Done when
- [ ] Every test cell prints ✅.
- [ ] You can explain *why* `x in some_set` is fast and `x in some_list` is slow.
- [ ] `../work/week03/vocab/` is a package with at least 6 passing tests that you wrote.
- [ ] You can write binary search from a blank page, with its loop invariant explained in a comment.

---
# Day 1: Classes (2.5 h)

A **class** bundles *data* (attributes) with the *behaviour* (methods) that works on that data. You've already used many classes: `list`, `dict`, `str`. Later, every PyTorch model you write will be a class.

- `__init__` runs when you create an object and sets its starting state.
- `self` is the object itself. Methods read and change `self.something`.
- Methods whose names start and end with double underscores (`__len__`, `__repr__`) are "dunder" methods. They let your object work with built-ins such as `len(obj)` and `print(obj)`.

In [ ]:
class RunningStats:
    """Keeps the count and mean of numbers seen so far, without storing them all."""

    def __init__(self):
        self.count = 0
        self.mean = 0.0

    def add(self, x):
        self.count += 1
        # incremental mean: new_mean = old_mean + (x - old_mean) / n
        self.mean += (x - self.mean) / self.count

    def __repr__(self):
        return f"RunningStats(count={self.count}, mean={self.mean:.3f})"


rs = RunningStats()
for x in [2, 4, 9]:
    rs.add(x)
print(rs)

✍️ *Derive the incremental-mean update from mean_n = (sum of n numbers) / n. Why is it useful when the data doesn't fit in memory?* (double-click to edit)



### Exercise 3.1: `Vocabulary`
Every NLP model turns words into integer ids. Build a class that does this:
- `Vocabulary()` starts with exactly one token, `"<unk>"` (for "unknown"), with id `0`.
- `add(token)` gives the token the next free id **if it's new**, and returns the token's id either way.
- `token_to_id(token)` returns the id, or `0` for unknown tokens.
- `id_to_token(i)` returns the token for an id.
- `len(vocab)` gives the number of tokens, including `<unk>`.
- `encode(tokens)` returns a list of ids; `decode(ids)` returns a list of tokens.

Use **two dicts** so that lookups in both directions are fast.

In [ ]:
class Vocabulary:
    UNK = "<unk>"

    def __init__(self):
        raise NotImplementedError

In [ ]:
v = Vocabulary()
assert len(v) == 1 and v.token_to_id("<unk>") == 0
assert v.add("lung") == 1
assert v.add("ct") == 2
assert v.add("lung") == 1          # already present, so the same id comes back
assert len(v) == 3
assert v.token_to_id("brain") == 0
assert v.id_to_token(2) == "ct"
assert v.encode(["ct", "lung", "brain"]) == [2, 1, 0]
assert v.decode([1, 2]) == ["lung", "ct"]
print("✅ 3.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

In `__init__`: `self.tok2id = {self.UNK: 0}` and `self.id2tok = {0: self.UNK}`.

</details>

<details><summary>Hint 2</summary>

In `add`: if the token is already in `self.tok2id`, return its id. Otherwise the new id is `len(self.tok2id)`; store it in both dicts.

</details>

<details><summary>Hint 3</summary>

`__len__(self)` must return an `int`. `encode` is a list comprehension over `token_to_id`.

</details>


### Dataclasses
When a class mostly *holds data*, `@dataclass` writes `__init__`, `__repr__` and `__eq__` for you.

In [ ]:
from dataclasses import dataclass

@dataclass
class Document:
    doc_id: int
    title: str
    text: str

    def n_words(self):
        return len(self.text.split())

d = Document(1, "Pneumonia detection", "Deep learning detects pneumonia in X-rays")
print(d, d.n_words())
print(d == Document(1, "Pneumonia detection", "Deep learning detects pneumonia in X-rays"))

---
# Day 2: Complexity by experiment (2.5 h)

**Big-O** describes how running time *grows* as the input size $n$ grows, ignoring constant factors.

| Big-O | Name | Example | Going from n = 1,000 to 1,000,000 |
|---|---|---|---|
| $O(1)$ | constant | `x in set`, `d[key]` | same time |
| $O(\log n)$ | logarithmic | binary search | about 2× |
| $O(n)$ | linear | `x in list`, one loop | 1,000× |
| $O(n \log n)$ | "n log n" | `sorted()` | about 2,000× |
| $O(n^2)$ | quadratic | two nested loops over the data | 1,000,000× |

Don't just believe the table: **measure it**.

In [ ]:
import time, random
import matplotlib.pyplot as plt

def time_membership(n, container_type, repeats=200):
    data = list(range(n))
    container = set(data) if container_type == "set" else data
    targets = [random.randrange(n) for _ in range(repeats)]
    start = time.perf_counter()
    for t in targets:
        _ = t in container
    return (time.perf_counter() - start) / repeats

sizes = [1_000, 5_000, 20_000, 50_000, 100_000]
list_times = [time_membership(n, "list") for n in sizes]
set_times = [time_membership(n, "set") for n in sizes]

plt.plot(sizes, list_times, "o-", label="list: O(n)")
plt.plot(sizes, set_times, "o-", label="set: O(1)")
plt.xlabel("n"); plt.ylabel("seconds per lookup"); plt.legend(); plt.title("Membership test")
plt.show()

✍️ *Describe the two curves. A set is built on a hash table: roughly, how does it find an item without looking through everything?* (double-click to edit)



### Exercise 3.2: Classify these snippets
Give the big-O of each snippet in terms of `n = len(items)`, *then* open the answers.
```python
# (a)
total = sum(items)
# (b)
for i in range(len(items)):
    for j in range(i + 1, len(items)):
        pass
# (c)
items.sort(); first = items[0]
# (d)
seen = set()
for x in items:
    seen.add(x)
# (e)
for x in items:
    if x in items:      # careful!
        pass
```

✍️ *Your answer:* (double-click to edit)



<details><summary>Answers (open only after you've written yours)</summary>

(a) O(n). (b) O(n²): about n²/2 pairs, and constants are dropped. (c) O(n log n), which is the cost of the sort. (d) O(n): n set insertions at O(1) each. (e) O(n²): a hidden linear search inside a loop. This is the most common performance bug in beginners' AI code.
</details>

### Exercise 3.3: `has_duplicates`
The slow version is given. Write `has_duplicates_fast` in **O(n)**, then time both versions at n = 5,000.

In [ ]:
def has_duplicates_slow(items):
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if items[i] == items[j]:
                return True
    return False

def has_duplicates_fast(items):
    raise NotImplementedError

In [ ]:
for case in [[1, 2, 3], [1, 2, 1], [], ["a", "a"]]:
    assert has_duplicates_fast(case) == has_duplicates_slow(case)
big = list(range(5000))
t0 = time.perf_counter(); has_duplicates_slow(big); t1 = time.perf_counter()
has_duplicates_fast(big); t2 = time.perf_counter()
print(f"slow {t1 - t0:.3f}s, fast {t2 - t1:.5f}s, speed-up ×{(t1 - t0) / max(t2 - t1, 1e-9):.0f}")
print("✅ 3.3 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Keep a `seen` set. For each item, if it's already in `seen`, return `True`; otherwise add it.

</details>


### Exercise 3.4: `two_sum` (a classic interview problem)
Return indices `(i, j)` with `i < j` such that `nums[i] + nums[j] == target`, or `None` if there are none. Aim for **O(n)** using a dict.

In [ ]:
def two_sum(nums, target):
    raise NotImplementedError

In [ ]:
assert two_sum([2, 7, 11, 15], 9) == (0, 1)
assert two_sum([3, 2, 4], 6) == (1, 2)
assert two_sum([1, 2], 10) is None
assert two_sum([3, 3], 6) == (0, 1)
print("✅ 3.4 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

For each number `x` at index `j`, the partner you need is `target - x`.

</details>

<details><summary>Hint 2</summary>

Keep a dict `value -> index` of the numbers you've already seen. Check it *before* adding `x`.

</details>


---
# Day 3: Binary search, insertion sort, randomised testing (2.5 h)

### Binary search
In a **sorted** list, compare the target with the middle item, then throw away the half that can't contain it. Each step halves the search space, so it takes about $\log_2 n$ steps: around 20 steps for a million items.

The key idea is a **loop invariant**: something that stays true on every pass of the loop. Here it's: *"if the target is in the list, it is somewhere in `items[lo..hi]`."* Each update must keep this true, and the loop stops when the range is empty.

### Exercise 3.5: `binary_search`
Return the index of `target`, or `-1`. Write the invariant as a comment in your code.

In [ ]:
def binary_search(items, target):
    raise NotImplementedError

### Randomised testing
Hand-written tests only check the cases *you* thought of. A **randomised test** compares your function with a simple, obviously correct reference on hundreds of random inputs. It's one of the most effective ways to find bugs.

In [ ]:
import random
random.seed(0)                     # same random inputs every run: reproducible
for _ in range(500):
    items = sorted(random.sample(range(100), random.randint(0, 20)))
    target = random.randrange(100)
    expected = items.index(target) if target in items else -1
    got = binary_search(items, target)
    assert got == expected, f"items={items}, target={target}: expected {expected}, got {got}"
print("✅ 3.5 passed 500 random tests")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Start with `lo, hi = 0, len(items) - 1` and loop `while lo <= hi:`.

</details>

<details><summary>Hint 2</summary>

`mid = (lo + hi) // 2`. If `items[mid] < target`, the target can only be to the right: `lo = mid + 1`.

</details>

<details><summary>Hint 3</summary>

If the loop ends without finding it, return `-1`. An infinite loop usually means you wrote `lo = mid` instead of `mid + 1`.

</details>


### Exercise 3.6: `insertion_sort`
Return a **new** sorted list without changing the input. Insertion sort keeps a sorted part on the left and *inserts* each new item into its place, the way you'd sort playing cards in your hand. Don't use `sorted` or `.sort`.

In [ ]:
def insertion_sort(items):
    raise NotImplementedError

In [ ]:
random.seed(1)
for _ in range(300):
    data = [random.randint(-50, 50) for _ in range(random.randint(0, 15))]
    original = list(data)
    assert insertion_sort(data) == sorted(data)
    assert data == original, "input was modified"
print("✅ 3.6 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Copy first: `out = list(items)`. Then for `i` from 1 to the end, take `key = out[i]`.

</details>

<details><summary>Hint 2</summary>

Move left with `j = i - 1`. While `j >= 0 and out[j] > key`, shift `out[j]` one place right and decrease `j`. Then put `key` at `j + 1`.

</details>


✍️ *What's the big-O of insertion sort in the worst case, and in the best case? Which input gives the best case?* (double-click to edit)



---
# Day 4: Package layout (2 h)

Real projects separate **source code** from **tests**. Create this structure in VS Code:
```
work/week03/
├── vocab/
│   ├── __init__.py          # can be empty; it marks the folder as a package
│   └── vocabulary.py        # your Vocabulary class
└── tests/
    └── test_vocabulary.py   # from vocab.vocabulary import Vocabulary
```
### Exercise 3.7
Move `Vocabulary` into `vocab/vocabulary.py`, add docstrings, and write **at least 6 tests** in `tests/test_vocabulary.py`. Add one *new* feature, test-first:
- `Vocabulary.from_tokens(tokens, min_count=1)`, a `@classmethod` that builds a vocabulary containing only the tokens that appear at least `min_count` times, added in order of first appearance. (Rare words are often mapped to `<unk>`; this is how real tokenisers keep the vocabulary small.)

**Test-first** means: write the test for `from_tokens`, watch it fail, *then* implement it.

Run tests from the terminal with `cd work/week03 && python -m pytest -q`, or use the checker below.

In [ ]:
# Create the files in VS Code, then run the checker.

In [ ]:
import subprocess, sys, importlib
r = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd="../work/week03", capture_output=True, text=True)
print(r.stdout[-2000:])
assert r.returncode == 0, "tests fail"
n = open("../work/week03/tests/test_vocabulary.py").read().count("def test_")
assert n >= 6, f"only {n} tests"
sys.path.insert(0, "../work/week03")
from vocab import vocabulary
importlib.reload(vocabulary)
V = vocabulary.Vocabulary.from_tokens(["a", "b", "a", "c", "c", "c"], min_count=2)
assert V.encode(["a", "b", "c"]) == [1, 0, 2]
print(f"✅ 3.7 passed with {n} tests")

---
# Review (30 min)

### Blank-page rebuild
Write `binary_search` again from memory, with the invariant comment. Then reuse the randomised test above on it.

In [ ]:
def binary_search2(items, target):
    raise NotImplementedError

random.seed(2)
for _ in range(300):
    items = sorted(random.sample(range(60), random.randint(0, 15)))
    t = random.randrange(60)
    assert binary_search2(items, t) == (items.index(t) if t in items else -1)
print("✅ rebuild passed")

### Spaced recall
1. *(Week 1)* What does `string.punctuation` contain, and how did you use it?
2. *(Week 2)* Write, from memory, the pytest pattern for checking that a function raises `ValueError`.
3. *(Week 2)* What is a dict comprehension? Give an example.
4. What are `self` and `__init__`?
5. Why does `x in list` inside a loop make code quadratic?
6. What's the loop invariant of binary search?
7. Why is randomised testing against a simple reference so effective?

✍️ *Your answer:* (double-click to edit)



### Commit, log, send
Use the branch, PR and merge workflow, update `LOG.md`, and **send me** the PR link.